# Statistical Analysis
## Public Compliance Data Analysis - MBA Thesis

**Objective:** Perform rigorous statistical analysis:
- Correlation analysis
- Hypothesis testing
- Multiple linear regression
- Model diagnostics and validation

In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


## Step-by-step (Aula style)

1. Packages and environment setup
2. Reproducibility
3. Data loading
4. Analysis blocks
5. Summary and interpretation


# Packages


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from scipy.stats import shapiro, normaltest, jarque_bera

# Try local loader first, fallback to S3 if available
from src.analysis.local_data_loader import LocalGoldDataLoader as GoldDataLoader

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)
pd.set_option('display.float_format', '{:.4f}'.format)


In [ ]:
import matplotlib as mpl
mpl.rcParams['axes.formatter.useoffset'] = False
mpl.rcParams['axes.formatter.limits'] = (-99, 99)


# Reproducibility


In [ ]:
import os
import random

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Reproducibility seed fixed at {SEED}")


In [ ]:
import json as _json

_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}

S3_BUCKET_NAME = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))
AWS_PROFILE = os.environ.get('AWS_PROFILE', _rtcfg.get('aws', {}).get('profile', None))


## 1. Load Data

In [ ]:
# Try local loader first, fallback to S3 if available
from src.analysis.local_data_loader import LocalGoldDataLoader as GoldDataLoader
loader = GoldDataLoader()

# --- Municipality-level analysis (N ~= 5,570) -----------------------------
# We now load the municipality-level `analysis_compliance_municipality` dataset
# instead of the state-level `analysis_compliance` (N=27). The municipality
# grain gives proper statistical power for correlation, OLS and ML work below.
#
# For backward compatibility with the rest of the notebook we:
#   1. Rename muni-level columns to the old state-level names (e.g.
#      `population_2022` -> `population`) so downstream cells work unchanged.
#   2. Regenerate the human-readable region dummies (is_norte, is_nordeste,
#      is_sudeste, is_sul, is_centro_oeste) with the same names they had in
#      the state-level dataset.
#   3. Add state dummies (is_state_<IBGE-2digit-code>) as extra features.
df = loader.load_dataset('analysis_compliance_municipality')
df = df.rename(columns={
    'population_2022': 'population',
    'literacy_rate_2022': 'avg_literacy_rate',
    'avg_income_2022': 'avg_income',
})

REGION_NAME_TO_DUMMY = {
    'Norte': 'is_norte',
    'Nordeste': 'is_nordeste',
    'Sudeste': 'is_sudeste',
    'Sul': 'is_sul',
    'Centro-Oeste': 'is_centro_oeste',
}
for _rname, _col in REGION_NAME_TO_DUMMY.items():
    df[_col] = (df['region_name'] == _rname).astype('Int64')
REGION_DUMMY_COLS = list(REGION_NAME_TO_DUMMY.values())

state_dummies = pd.get_dummies(df['state_code'], prefix='is_state').astype('Int64')
df = pd.concat([df, state_dummies], axis=1)
STATE_DUMMY_COLS = list(state_dummies.columns)

print(f"Loaded {len(df):,} observations (municipalities across {df['state_code'].nunique()} states)")
print(f"Region dummies: {REGION_DUMMY_COLS}")
print(f"State dummies: {len(STATE_DUMMY_COLS)} columns (first: {STATE_DUMMY_COLS[0]}, last: {STATE_DUMMY_COLS[-1]})")
df.head()

## 2. Correlation Analysis

### 2.1 Pearson Correlation Matrix

In [ ]:
# Key analytical variables (municipality level). Dropped `n_municipalities`
# because it is constant at this grain. Added `log_total_transfers` where
# available to expose the transfer-side signal (central to the thesis question).
key_vars = ['sanctions_per_100k', 'avg_literacy_rate', 'avg_income',
            'log_population', 'log_income']
if 'log_total_transfers' in df.columns:
    key_vars.append('log_total_transfers')

# Cast to plain float -- Int64/Float64 nullable types with NaN can trip up
# seaborn heatmap rendering.
corr_matrix = df[key_vars].astype('Float64').astype(float).corr(method='pearson')

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Pearson Correlation Matrix (municipality-level, N=5,570)',
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\nCorrelations with Sanctions per 100k:")
print("=" * 60)
sanctions_corr = corr_matrix['sanctions_per_100k'].sort_values(ascending=False)
print(sanctions_corr)

### 2.2 Statistical Significance Testing

In [ ]:
from scipy.stats import pearsonr

def correlation_test(x, y, var_names):
    """Compute Pearson correlation with p-value."""
    clean_data = pd.DataFrame({'x': x, 'y': y}).dropna()
    if len(clean_data) < 3:
        return None, None
    r, p = pearsonr(clean_data['x'].astype(float), clean_data['y'].astype(float))
    return r, p

results = []
target = df['sanctions_per_100k']

_test_vars = ['avg_literacy_rate', 'avg_income', 'log_income', 'log_population']
if 'log_total_transfers' in df.columns:
    _test_vars.append('log_total_transfers')

for var in _test_vars:
    r, p = correlation_test(target, df[var], (var, 'sanctions_per_100k'))
    if r is not None:
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        results.append({
            'Variable': var,
            'Correlation (r)': f"{r:.4f}",
            'P-value': f"{p:.4f}",
            'Significance': sig,
        })

print("Pearson correlation tests vs. sanctions_per_100k (N=5,570)")
print("=" * 60)
print(pd.DataFrame(results).to_string(index=False))

### 2.3 Scatter Plots with Regression Lines

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

vars_to_plot = [
    ('avg_literacy_rate', 'Literacy Rate (%)', axes[0, 0]),
    ('avg_income', 'Average Income (BRL)', axes[0, 1]),
    ('log_income', 'Log(Income)', axes[1, 0]),
    ('log_population', 'Log(Population)', axes[1, 1])
]

for var, label, ax in vars_to_plot:
    ax.scatter(df[var], df['sanctions_per_100k'], alpha=0.6, s=100)
    
    z = np.polyfit(df[var].dropna(), df['sanctions_per_100k'][df[var].notna()], 1)
    p = np.poly1d(z)
    ax.plot(df[var].sort_values(), p(df[var].sort_values()), "r--", alpha=0.8, linewidth=2)
    
    r, pval = correlation_test(df[var], df['sanctions_per_100k'], (var, 'sanctions'))
    ax.text(0.05, 0.95, f'r = {r:.3f}\np = {pval:.4f}', 
            transform=ax.transAxes, fontsize=11, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlabel(label, fontsize=12)
    ax.set_ylabel('Sanctions per 100k', fontsize=12)
    ax.set_title(f'Sanctions vs {label}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Hypothesis Testing

### 3.1 Regional Differences (ANOVA)

In [ ]:
from scipy.stats import f_oneway

regions = df['region_name'].unique()
groups = [df[df['region_name'] == region]['sanctions_per_100k'].dropna() for region in regions]

f_stat, p_value = f_oneway(*groups)

print("One-Way ANOVA: Regional Differences in Sanctions per 100k")
print("=" * 70)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"\nResult: {'Significant' if p_value < 0.05 else 'Not significant'} regional differences")
print("\nRegional Means:")
print(df.groupby('region_name')['sanctions_per_100k'].agg(['mean', 'std', 'count']).round(2))


In [ ]:
plt.figure(figsize=(12, 6))
df.boxplot(column='sanctions_per_100k', by='region_name', ax=plt.gca())
plt.title('Sanctions per 100k by Region', fontsize=14, fontweight='bold')
plt.suptitle('')
plt.xlabel('Region', fontsize=12)
plt.ylabel('Sanctions per 100k', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 3.2 Post-hoc Tests (Tukey HSD)

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(endog=df['sanctions_per_100k'], 
                          groups=df['region_name'], 
                          alpha=0.05)

print("Tukey HSD Post-hoc Test")
print("=" * 80)
print(tukey)

tukey.plot_simultaneous()
plt.tight_layout()
plt.show()


## 4. Multiple Linear Regression

### 4.1 Model Specification

In [ ]:
y = df['sanctions_per_100k']

# Drop `is_sudeste` as the reference region to avoid the dummy trap.
X_vars = ['log_income', 'avg_literacy_rate', 'log_population',
          'is_norte', 'is_nordeste', 'is_sul', 'is_centro_oeste']
if 'log_total_transfers' in df.columns:
    X_vars.append('log_total_transfers')

# Drop rows with any NaN in the selected features (muni-level has some
# nullable columns) and convert to plain float for statsmodels.
_mask = df[X_vars + ['sanctions_per_100k']].notna().all(axis=1)
X = df.loc[_mask, X_vars].copy().astype(float)
y = df.loc[_mask, 'sanctions_per_100k'].astype(float)
X = sm.add_constant(X)

print("Model Specification:")
print("=" * 70)
print(f"Dependent Variable: sanctions_per_100k (municipality level)")
print(f"Independent Variables: {X_vars}")
print(f"Reference region (omitted to avoid dummy trap): Sudeste")
print(f"\nSample size: {len(X):,} municipalities (rows with any NaN dropped)")
print(f"Number of predictors: {len(X_vars)}")

### 4.2 OLS Regression

In [ ]:
model = sm.OLS(y, X).fit()

print(model.summary())


### 4.3 Coefficient Interpretation

In [ ]:
coef_df = pd.DataFrame({
    'Variable': model.params.index,
    'Coefficient': model.params.values,
    'Std Error': model.bse.values,
    't-statistic': model.tvalues.values,
    'P-value': model.pvalues.values,
    'CI Lower': model.conf_int()[0].values,
    'CI Upper': model.conf_int()[1].values
})

coef_df['Significant'] = coef_df['P-value'].apply(
    lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
)

print("\nRegression Coefficients")
print("=" * 100)
display(coef_df.round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

coef_plot = coef_df[coef_df['Variable'] != 'const'].copy()
coef_plot = coef_plot.sort_values('Coefficient')

colors = ['red' if p < 0.05 else 'gray' for p in coef_plot['P-value']]

ax.barh(coef_plot['Variable'], coef_plot['Coefficient'], color=colors, alpha=0.7)
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Coefficient Value', fontsize=12)
ax.set_title('Regression Coefficients (Red = Significant at p<0.05)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


## 5. Model Diagnostics

### 5.1 Residual Analysis

In [ ]:
residuals = model.resid
fitted = model.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].scatter(fitted, residuals, alpha=0.6)
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot', fontweight='bold')

axes[1, 0].hist(residuals, bins=15, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Histogram of Residuals', fontweight='bold')
axes[1, 0].axvline(x=0, color='r', linestyle='--')

standardized_resid = residuals / np.std(residuals)
axes[1, 1].scatter(fitted, np.sqrt(np.abs(standardized_resid)), alpha=0.6)
axes[1, 1].set_xlabel('Fitted Values')
axes[1, 1].set_ylabel('√|Standardized Residuals|')
axes[1, 1].set_title('Scale-Location Plot', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 5.2 Normality Tests

In [ ]:
shapiro_stat, shapiro_p = shapiro(residuals)
jb_stat, jb_p = jarque_bera(residuals)

print("Normality Tests for Residuals")
print("=" * 70)
print(f"Shapiro-Wilk Test:")
print(f"  Statistic: {shapiro_stat:.4f}")
print(f"  P-value: {shapiro_p:.4f}")
print(f"  Result: {'Reject normality' if shapiro_p < 0.05 else 'Cannot reject normality'}\n")

print(f"Jarque-Bera Test:")
print(f"  Statistic: {jb_stat:.4f}")
print(f"  P-value: {jb_p:.4f}")
print(f"  Result: {'Reject normality' if jb_p < 0.05 else 'Cannot reject normality'}")


### 5.3 Heteroskedasticity Tests

In [ ]:
bp_stat, bp_p, _, _ = het_breuschpagan(residuals, X)

print("Heteroskedasticity Tests")
print("=" * 70)
print(f"Breusch-Pagan Test:")
print(f"  Statistic: {bp_stat:.4f}")
print(f"  P-value: {bp_p:.4f}")
print(f"  Result: {'Heteroskedasticity detected' if bp_p < 0.05 else 'Homoskedasticity (constant variance)'}")


### 5.4 Multicollinearity (VIF)

In [ ]:
X_no_const = X.drop('const', axis=1)

vif_data = pd.DataFrame()
vif_data['Variable'] = X_no_const.columns
vif_data['VIF'] = [variance_inflation_factor(X_no_const.values, i) 
                   for i in range(X_no_const.shape[1])]

vif_data['Multicollinearity'] = vif_data['VIF'].apply(
    lambda x: 'High (>10)' if x > 10 else 'Moderate (5-10)' if x > 5 else 'Low (<5)'
)

print("Variance Inflation Factor (VIF) Analysis")
print("=" * 70)
print("Rule of thumb: VIF > 10 indicates high multicollinearity\n")
display(vif_data.sort_values('VIF', ascending=False))


### 5.5 Influential Observations

In [ ]:
from statsmodels.stats.outliers_influence import OLSInfluence

influence = OLSInfluence(model)
cooks_d = influence.cooks_distance[0]

fig, ax = plt.subplots(figsize=(14, 6))
ax.stem(range(len(cooks_d)), cooks_d, markerfmt=',')
ax.axhline(y=4/len(X), color='r', linestyle='--', label='Threshold (4/n)')
ax.set_xlabel('Observation Index')
ax.set_ylabel("Cook's Distance")
ax.set_title("Cook's Distance - Influential Observations", fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

threshold = 4 / len(X)
influential = df[cooks_d > threshold][['state_name', 'sanctions_per_100k']]
if len(influential) > 0:
    print(f"\nInfluential observations (Cook's D > {threshold:.4f}):")
    print(influential)
else:
    print("\nNo highly influential observations detected.")


## 6. Model Comparison

In [ ]:
# Two-variable baseline for comparison (same sample size as the full model).
_simple_vars = ['log_income', 'avg_literacy_rate']
X_simple = sm.add_constant(df.loc[_mask, _simple_vars].astype(float))
model_simple = sm.OLS(y, X_simple).fit()

model_full = model

comparison = pd.DataFrame({
    'Model': [f'Simple ({len(_simple_vars)} vars)', f'Full ({len(X_vars)} vars inc. region dummies)'],
    'N': [int(model_simple.nobs), int(model_full.nobs)],
    'R-squared': [model_simple.rsquared, model_full.rsquared],
    'Adj. R-squared': [model_simple.rsquared_adj, model_full.rsquared_adj],
    'AIC': [model_simple.aic, model_full.aic],
    'BIC': [model_simple.bic, model_full.bic],
    'F-statistic': [model_simple.fvalue, model_full.fvalue],
    'Prob (F)': [model_simple.f_pvalue, model_full.f_pvalue],
})

print("Model Comparison (municipality level)")
print("=" * 80)
display(comparison.round(4))

print("\nNote: Lower AIC/BIC indicates better model fit")

## 7. Summary of Findings

### 7.1 Correlation Analysis
- **Average income** is the strongest predictor of sanctions per 100k (r = 0.74, p < 0.001).
- **Log-transformed income** also shows strong correlation (r = 0.63).
- **Literacy rate** has a moderate positive association (r = 0.47).
- Population size and number of municipalities show weak negative associations.

### 7.2 Regional Differences
- **One-way ANOVA** found no statistically significant differences between regions (F = 1.70, p = 0.186).
- **Tukey HSD** post-hoc tests confirm no pairwise regional comparisons are significant at α = 0.05.
- Centro-Oeste has the highest mean (25.66) but also the highest variance (std = 26.74), driven by the Distrito Federal outlier.

### 7.3 Regression Results
- **OLS regression** achieves R² = 0.835 (Adj. R² = 0.775), explaining most of the variance in sanctions rates.
- **Log income** is the strongest significant predictor (β = 49.75, p < 0.001): a 1% increase in average income is associated with ~0.50 more sanctions per 100k.
- **Regional dummies** (Norte: β = 20.94, p = 0.003; Nordeste: β = 22.97, p = 0.009) are significant, indicating that after controlling for income, these regions have higher-than-expected sanctions rates.
- **Literacy rate** is not significant when income is already in the model (p = 0.988), suggesting its bivariate correlation is mediated by income.

### 7.4 Model Diagnostics
- **Normality**: Shapiro-Wilk (p = 0.094) and Jarque-Bera (p = 0.340) — residuals are approximately normal.
- **Homoskedasticity**: Breusch-Pagan test (p = 0.299) — no evidence of heteroskedasticity.
- **Influential observations**: Distrito Federal (Cook's D > threshold) is the most influential point, along with Tocantins and Paraná.

### 7.5 Limitations
- Small sample size (n = 27 states) limits statistical power and the number of predictors.
- Cross-sectional design cannot establish causality — higher income may correlate with greater institutional capacity to detect and record sanctions, not with more actual misconduct.
- The Distrito Federal is a structural outlier (federal capital with unique governance) that strongly influences results.
